# Notebook 5 — LSTM-DANN Domain Adaptation

**Goal:** Train the LSTM Domain Adversarial Neural Network (LSTM-DANN)
to learn machine-independent RUL features.

**Architecture** (Section 3.4 of the paper):
- Shared LSTM Feature Extractor g_f → produces domain-invariant embedding f
- RUL Regressor g_y: f → predicted RUL (minimises MAE on SOURCE labels)
- Domain Classifier g_d: f → GRL → domain binary classifier
  (minimises BCE on source/target labels; GRL reverses gradient for g_f)

**Training** (Section 5.1):
- Two-pass SGD: regression pass on source, adversarial pass on source+target
- Early stopping on source validation MAE
- LR decay ×0.1 at epoch 100


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import os

from src.data_loader    import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor   import full_preprocess_pipeline
from src.windowing      import create_windows, create_windows_inference
from src.models.lstm_dann import build_lstm_dann, get_feature_extractor
from src.train          import LSTMDANNTrainer
from src.evaluate       import rmse, evaluate_model

tf.random.set_seed(42)
np.random.seed(42)

WINDOW_SIZE = 30
MAX_RUL     = 125

# Load and preprocess all datasets
datasets = load_all_datasets(data_dir='../data/raw')
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    df_tr, df_te, scaler = full_preprocess_pipeline(
        df_train=datasets[ds_id]['train'],
        df_test=datasets[ds_id]['test'],
        feature_cols=FEATURE_COLS, sensor_cols=SENSOR_COLS,
        smooth=True, max_rul=MAX_RUL,
        scaler_save_path=f'../models/saved/scaler_{ds_id}.joblib'
    )
    datasets[ds_id]['train_norm'] = df_tr
    datasets[ds_id]['test_norm']  = df_te
    X, y, _ = create_windows(df_tr, FEATURE_COLS, WINDOW_SIZE)
    datasets[ds_id]['X_train'] = X
    datasets[ds_id]['y_train'] = y

print("Preprocessing complete.")


## 5.1 Architecture Overview


In [ ]:
_, dann_model_demo = build_lstm_dann(
    window_size=WINDOW_SIZE, n_features=len(FEATURE_COLS),
    lstm_units=128, lstm_layers=1, feature_dim=64,
    reg_units=[32], domain_units=[32],
    lstm_dropout=0.5, reg_dropout=0.3, dom_dropout=0.3, alpha=0.8
)
dann_model_demo.summary()


**Architecture components:**

| Sub-network | Layers | Purpose |
|------------|--------|---------|
| Feature Extractor g_f | LSTM(128) → Dropout(0.5) → Dense(64, ReLU) | Extract temporal patterns from sensor windows |
| RUL Regressor g_y | Dense(32, ReLU) → Dropout(0.3) → Dense(1) | Map features → RUL scalar |
| Domain Classifier g_d | GRL(α=0.8) → Dense(32, ReLU) → Dropout(0.3) → Dense(1, Sigmoid) | Distinguish source from target (adversarially) |

The GRL ensures g_f is trained adversarially: while g_d tries to tell domains
apart, g_f learns to produce features that make this IMPOSSIBLE.


## 5.2 Training: FD001 → FD002 (Primary Example)

Hyperparameters from Table 3 of the paper (Source FD001, Target FD002 row).


In [ ]:
SOURCE_DS = 'FD001'
TARGET_DS = 'FD002'

X_src = datasets[SOURCE_DS]['X_train'].astype(np.float32)
y_src = datasets[SOURCE_DS]['y_train'].astype(np.float32)
X_tgt = datasets[TARGET_DS]['X_train'].astype(np.float32)

X_src_tr, X_src_val, y_src_tr, y_src_val = train_test_split(
    X_src, y_src, test_size=0.1, random_state=42
)

print(f"Source train: {X_src_tr.shape} | Source val: {X_src_val.shape}")
print(f"Target train: {X_tgt.shape}")


In [ ]:
# Build model with paper hyperparameters for FD001 → FD002
reg_model_01_02, dann_model_01_02 = build_lstm_dann(
    window_size=WINDOW_SIZE, n_features=len(FEATURE_COLS),
    lstm_units=128, lstm_layers=1, feature_dim=64,
    reg_units=[32], domain_units=[32],
    lstm_dropout=0.5, reg_dropout=0.3, dom_dropout=0.3,
    alpha=0.8
)

trainer = LSTMDANNTrainer(
    dann_model_01_02, alpha=0.8,
    lr_reg=0.01, lr_dom=0.01
)

history = trainer.fit(
    X_src_tr, y_src_tr, X_tgt,
    X_val_src=X_src_val, y_val_src=y_src_val,
    epochs=200, batch_size=256,
    patience=20, lr_decay_epoch=100
)

dann_model_01_02.save_weights(
    f'../models/saved/lstm_dann_{SOURCE_DS}_to_{TARGET_DS}.weights.h5'
)
print("Model weights saved.")


## 5.3 Training Curve Analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['rul_loss'], color='steelblue', linewidth=2)
axes[0].set_title(f'RUL Regression Loss\n({SOURCE_DS} source domain)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(history['dom_loss'], color='coral', linewidth=2)
axes[1].axhline(np.log(2), color='gray', linestyle='--', linewidth=1.5,
                label=f'Random guess = ln(2) ≈ {np.log(2):.3f}')
axes[1].set_title('Domain Classification Loss\n(Should converge near ln(2))')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Binary Cross-Entropy')
axes[1].legend()
axes[1].grid(alpha=0.3)

if history['val_mae']:
    axes[2].plot(history['val_mae'], color='seagreen', linewidth=2)
    axes[2].set_title(f'Validation MAE on {SOURCE_DS}\n(Early stopping criterion)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('MAE')
    axes[2].grid(alpha=0.3)

plt.suptitle(f'LSTM-DANN Training Curves: {SOURCE_DS} → {TARGET_DS}', fontsize=13)
plt.tight_layout()
plt.show()


**Reading the three curves:**

1. **RUL Regression Loss** (decreasing): The model learns to predict RUL
   from source data — this should drop and stabilise.

2. **Domain Classification Loss** (converging to ~0.693 = ln(2)):
   When this stabilises near the random-guess value, the domain classifier
   can no longer distinguish source from target. This confirms the feature
   extractor has learned truly domain-invariant representations.

3. **Validation MAE** (used for early stopping): The criterion from
   Section 5.2 of the paper — select hyperparameters giving lowest source
   RMSE while domain classification stabilises near random performance.


## 5.4 Domain Confusion Visualisation with t-SNE


In [ ]:
feature_extractor = get_feature_extractor(dann_model_01_02)

N_VIS = 500
feats_src = feature_extractor.predict(X_src[:N_VIS], verbose=0)
feats_tgt = feature_extractor.predict(X_tgt[:N_VIS], verbose=0)

all_feats  = np.vstack([feats_src, feats_tgt])
all_labels = ['Source (FD001)'] * N_VIS + ['Target (FD002)'] * N_VIS

tsne        = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_result = tsne.fit_transform(all_feats)

fig, ax = plt.subplots(figsize=(10, 8))
for label, color, marker in [
    ('Source (FD001)', 'steelblue', 'o'),
    ('Target (FD002)', 'coral',     's')
]:
    mask = [l == label for l in all_labels]
    ax.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
               c=color, label=label, alpha=0.4, s=15, marker=marker)

ax.set_title(f't-SNE of DANN Feature Embeddings\n{SOURCE_DS} (Source) vs {TARGET_DS} (Target)',
             fontsize=12)
ax.legend(fontsize=11)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


**Interpreting the t-SNE plot:**
- **Interleaved clusters** → Successful adaptation: features are
  domain-invariant, source and target patterns are mixed in latent space
- **Separated clusters** → Failed adaptation: the feature extractor still
  encodes domain-specific information that the classifier can exploit

After successful DANN training, source and target points should be
substantially overlapping, confirming the adversarial objective was achieved.


## 5.5 Train All 12 Source-Target Pairs

Following the paper's experimental design: each of the 4 datasets acts
as source, the remaining 3 as targets (4 × 3 = 12 experiments).


In [ ]:
# Hyperparameter table from Table 3 of the paper
HYPERPARAMS = {
    ('FD001', 'FD002'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[32],  lstm_dropout=0.5, alpha=0.8, lr_reg=0.01, lr_dom=0.01),
    ('FD001', 'FD003'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[32],  lstm_dropout=0.5, alpha=0.8, lr_reg=0.01, lr_dom=0.01),
    ('FD001', 'FD004'): dict(lstm_units=128, lstm_layers=1, feature_dim=64, reg_units=[32,32],domain_units=[32], lstm_dropout=0.5, alpha=1.0, lr_reg=0.01, lr_dom=0.1),
    ('FD002', 'FD001'): dict(lstm_units=64,  lstm_layers=1, feature_dim=64, reg_units=[32],  domain_units=[16,16],lstm_dropout=0.1,alpha=1.0, lr_reg=0.01, lr_dom=0.01),
    ('FD002', 'FD003'): dict(lstm_units=64,  lstm_layers=1, feature_dim=512,reg_units=[64,32],domain_units=[64,32],lstm_dropout=0.1,alpha=2.0,lr_reg=0.1, lr_dom=0.1),
    ('FD002', 'FD004'): dict(lstm_units=32,  lstm_layers=2, feature_dim=32, reg_units=[32],  domain_units=[16],  lstm_dropout=0.1, alpha=1.0, lr_reg=0.1, lr_dom=0.1),
    ('FD003', 'FD001'): dict(lstm_units=64,  lstm_layers=2, feature_dim=128,reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0,lr_reg=0.01,lr_dom=0.01),
    ('FD003', 'FD002'): dict(lstm_units=64,  lstm_layers=2, feature_dim=64, reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0,lr_reg=0.01,lr_dom=0.01),
    ('FD003', 'FD004'): dict(lstm_units=64,  lstm_layers=2, feature_dim=64, reg_units=[32,32],domain_units=[32,32],lstm_dropout=0.3,alpha=2.0,lr_reg=0.01,lr_dom=0.01),
    ('FD004', 'FD001'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0, lr_reg=0.01, lr_dom=0.01),
    ('FD004', 'FD002'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0, lr_reg=0.01, lr_dom=0.01),
    ('FD004', 'FD003'): dict(lstm_units=100, lstm_layers=1, feature_dim=30, reg_units=[20],  domain_units=[20],  lstm_dropout=0.5, alpha=1.0, lr_reg=0.01, lr_dom=0.01),
}

# Storage for cross-domain results
dann_reg_models = {}

for (src, tgt), hp in HYPERPARAMS.items():
    print(f"\nTraining DANN: {src} → {tgt}")
    X_s = datasets[src]['X_train'].astype(np.float32)
    y_s = datasets[src]['y_train'].astype(np.float32)
    X_t = datasets[tgt]['X_train'].astype(np.float32)
    X_s_tr, X_s_val, y_s_tr, y_s_val = train_test_split(
        X_s, y_s, test_size=0.1, random_state=42
    )

    reg_m, dann_m = build_lstm_dann(
        window_size=WINDOW_SIZE, n_features=len(FEATURE_COLS),
        lstm_units=hp['lstm_units'], lstm_layers=hp['lstm_layers'],
        feature_dim=hp['feature_dim'], reg_units=hp['reg_units'],
        domain_units=hp['domain_units'], lstm_dropout=hp['lstm_dropout'],
        reg_dropout=0.3, dom_dropout=0.3, alpha=hp['alpha']
    )

    trainer = LSTMDANNTrainer(
        dann_m, alpha=hp['alpha'],
        lr_reg=hp['lr_reg'], lr_dom=hp['lr_dom']
    )
    trainer.fit(
        X_s_tr, y_s_tr, X_t,
        X_val_src=X_s_val, y_val_src=y_s_val,
        epochs=200, batch_size=256,
        patience=20, lr_decay_epoch=100
    )

    dann_m.save_weights(
        f'../models/saved/lstm_dann_{src}_to_{tgt}.weights.h5'
    )
    dann_reg_models[(src, tgt)] = reg_m

print("\nAll 12 domain adaptation experiments complete.")
